# Research foundations

A local test bed for learning the package APIs and trying small synthetic ideas.
Begin with a question, change a named scenario, inspect the resulting objects and
tables, record findings, then decide what deserves a maintained package interface.

This notebook begins with the foundations exercised during MR 00. It will feed
later data-fetching, EDA, strategy-research and monitoring notebooks as working
content grows. Sections 1–15 use synthetic inputs. Section 16 downloads and verifies the actual Binance daily archive and inspects the selected hour; no strategy run is performed. Set `ALPHA_BINANCE_DOWNLOAD=0` for an offline foundations-only run.
The P1 source contract was revised with founder approval on 2026-09-15; its hour and economic rules remain unchanged; the adjustable cases below are
synthetic engineering scenarios, not tuning of its frozen hypothesis.

**Run:** from the repository root, use the command in `examples/README.md`.
The runner executes once by default, archives each attempt under `examples/runs/`,
and updates this notebook's displayed outputs after success. `--repeat 2` explicitly
requests two fresh-kernel runs. During interactive editing, restart and run all
before interpreting a result; use the runner to preserve an experiment snapshot.

## 1. Environment and display tools

Use the pinned interpreter and installed package. Presentation helpers stay local;
domain identities, clocks, decoding and financial operations come from the package.
Interactive use: select the repository's `.venv/bin/python` kernel, then **Restart
Kernel → Run All**. The notebook uses the installed package; it does not patch
`sys.path`. `nbclient` and `nbformat` belong to the separate execution runner and
need not be installed in the interactive kernel. The environment table records
only notebook tools actually present in that kernel.


In [1]:
# standard library imports
import hashlib
import json
import os
import platform
import subprocess
import sys
from datetime import UTC, datetime, timedelta
from decimal import ROUND_HALF_UP, Decimal
from html import escape
from importlib.metadata import distributions, version
from pathlib import Path
from uuid import UUID

# third party library imports
from IPython.display import HTML, Markdown, display

# project library imports
import alpha_research
from alpha_research.data import BinancePayloadDecoder, TardisDecoder
from alpha_research.data.events import Disconnect, MarketEventError
from alpha_research.data.venues.binance.events import Trade
from alpha_research.financial_units import (
    Currency,
    Dimension,
    Money,
    Price,
    Quantity,
    QuantityKind,
    UnitError,
)
from alpha_research.instrument_identity import InstrumentId
from alpha_research.reference_data import CatalogSnapshot
from alpha_research.serialization import DomainCodec

root = Path.cwd().resolve()
if root.name == "examples":
    root = root.parent
assert (root / "configs/reference/binance-spot.json").is_file(), (
    "Run from the repository root or examples/."
)
notebook_path = Path(
    os.environ.get("ALPHA_RESEARCH_NOTEBOOK_PATH", root / "examples/research-foundations.ipynb")
).resolve()


class NotebookDisplay:
    """Render small tables for the exploratory notebook scenarios."""

    @staticmethod
    def show_table(headers: tuple[str, ...], rows: list[tuple[object, ...]]) -> None:
        """Display a small escaped table without adding a dataframe dependency.

        Args:
            headers: Column labels.
            rows: Exact or presentation values to render as text.
        """
        heading = "".join(
            f'<th style="text-align:left;padding:6px">{escape(label)}</th>' for label in headers
        )
        body = "".join(
            "<tr>"
            + "".join(
                f'<td style="padding:6px;vertical-align:top">{escape(str(value))}</td>' for value in row
            )
            + "</tr>"
            for row in rows
        )
        display(HTML(f"<table><thead><tr>{heading}</tr></thead><tbody>{body}</tbody></table>"))


## 2. Question and adjustable scenarios

Edit this cell for a new synthetic question. Change one factor at a time initially.
`exact_microsecond` changes capture precision; `smaller_quantity` changes trade size;
`unconfigured_symbol` explores a rejected source message. Add your own named cases.
The fixed regression examples in section 7 do not depend on these parameters.

In [2]:
QUESTION = (
    "How do capture precision, quantity and symbol selection affect "
    "decoded availability and exact notional?"
)
HYPOTHESIS = (
    "Submicrosecond capture is rounded upward, quantity changes notional exactly, "
    "and an unknown symbol is rejected."
)
EXPERIMENT_ID = "foundation-capture-and-units-v1"
SEED = 0  # Recorded for identity; this experiment uses no randomness.

INSTRUMENT_ID = "11111111-1111-4111-8111-111111111111"
SAMPLE_START = datetime(2024, 6, 1, tzinfo=UTC)
SAMPLE_END = SAMPLE_START + timedelta(hours=1)
LOCAL_RECEIVED = datetime(2026, 9, 11, tzinfo=UTC)
LOCAL_INGESTED = LOCAL_RECEIVED + timedelta(seconds=1)
EVENT_TIME_MS = 1717199999993
FOCUS_SCENARIO = "reference"
CUTOFF_OFFSETS_US = (-1, 0, 1, 1000000)

SCENARIOS = (
    {
        "name": "reference",
        "capture": "2024-06-01T00:00:00.0041023Z",
        "symbol": "BTCUSDT",
        "price": "67540.01000000",
        "quantity": "0.00146000",
    },
    {
        "name": "exact_microsecond",
        "capture": "2024-06-01T00:00:00.0041020Z",
        "symbol": "BTCUSDT",
        "price": "67540.01000000",
        "quantity": "0.00146000",
    },
    {
        "name": "smaller_quantity",
        "capture": "2024-06-01T00:00:00.0041023Z",
        "symbol": "BTCUSDT",
        "price": "67540.01000000",
        "quantity": "0.00100000",
    },
    {
        "name": "unconfigured_symbol",
        "capture": "2024-06-01T00:00:00.0041023Z",
        "symbol": "SOLUSDT",
        "price": "100.00",
        "quantity": "1.00",
    },
)

display(Markdown(f"**Question:** {QUESTION}\n\n**Expected mechanism:** {HYPOTHESIS}"))
NotebookDisplay.show_table(
    ("Scenario", "Capture text", "Symbol", "Price", "Quantity"),
    [
        (item["name"], item["capture"], item["symbol"], item["price"], item["quantity"])
        for item in SCENARIOS
    ],
)

**Question:** How do capture precision, quantity and symbol selection affect decoded availability and exact notional?

**Expected mechanism:** Submicrosecond capture is rounded upward, quantity changes notional exactly, and an unknown symbol is rejected.

Scenario,Capture text,Symbol,Price,Quantity
reference,2024-06-01T00:00:00.0041023Z,BTCUSDT,67540.01000000,0.00146000
exact_microsecond,2024-06-01T00:00:00.0041020Z,BTCUSDT,67540.01000000,0.00146000
smaller_quantity,2024-06-01T00:00:00.0041023Z,BTCUSDT,67540.01000000,0.00100000
unconfigured_symbol,2024-06-01T00:00:00.0041023Z,SOLUSDT,100.00,1.00


## 3. Identify the exact candidate and inputs

These checks compare installed runtime bytes with the source tree. If they fail,
refresh the frozen non-editable install and restart the kernel. A package path
alone is insufficient evidence that an edit is being exercised.

The runtime/input diff digest covers source, reference configuration and environment
inputs relative to HEAD. The notebook source digest excludes outputs, so refreshing
tables does not change the experiment's code identity. Successful runner snapshots
retain this table alongside inputs and findings; failed snapshots retain every cell
reached before the failure. These are local records, not the future formal trial store.

In [3]:
assert platform.python_version() == (root / ".python-version").read_text().strip() == "3.14.7"
installed = Path(alpha_research.__file__).resolve().parent
assert "site-packages" in installed.parts and installed != root / "src/alpha_research"
source_root = root / "src/alpha_research"
runtime_files = sorted(
    path
    for path in source_root.rglob("*")
    if path.is_file() and path.suffix in {".py", ".json", ".typed"}
)
assert runtime_files
runtime_digest = hashlib.sha256()
for source_file in runtime_files:
    relative = source_file.relative_to(source_root)
    source_bytes = source_file.read_bytes()
    assert source_bytes == (installed / relative).read_bytes(), f"Stale installed file: {relative}"
    runtime_digest.update(str(relative).encode() + b"\0" + source_bytes + b"\0")

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=root, text=True).strip()
branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=root, text=True).strip()
working_status = subprocess.check_output(["git", "status", "--short"], cwd=root, text=True).strip()
candidate_diff = subprocess.check_output(
    [
        "git",
        "diff",
        "--binary",
        "HEAD",
        "--",
        "src",
        "configs",
        "pyproject.toml",
        "uv.lock",
        ".python-version",
    ],
    cwd=root,
)
notebook_document = json.loads(notebook_path.read_text())
notebook_source = [
    (cell["cell_type"], "".join(cell["source"])) for cell in notebook_document["cells"]
]
notebook_digest = hashlib.sha256(
    json.dumps(notebook_source, ensure_ascii=False, separators=(",", ":")).encode()
).hexdigest()
experiment_inputs = {
    "experiment_id": EXPERIMENT_ID,
    "question": QUESTION,
    "hypothesis": HYPOTHESIS,
    "seed": SEED,
    "instrument_id": INSTRUMENT_ID,
    "sample_start": SAMPLE_START.isoformat(),
    "sample_end": SAMPLE_END.isoformat(),
    "local_received": LOCAL_RECEIVED.isoformat(),
    "local_ingested": LOCAL_INGESTED.isoformat(),
    "event_time_ms": EVENT_TIME_MS,
    "scenarios": SCENARIOS,
    "focus_scenario": FOCUS_SCENARIO,
    "cutoff_offsets_us": CUTOFF_OFFSETS_US,
}
input_digest = hashlib.sha256(json.dumps(experiment_inputs, sort_keys=True).encode()).hexdigest()
identity = {
    "notebook_path": str(notebook_path),
    "attempt_id": os.environ.get("ALPHA_RESEARCH_ATTEMPT_ID", "interactive: unsnapshotted"),
    "commit": commit,
    "branch": branch,
    "working_tree": "changed" if working_status else "clean",
    "python": platform.python_version(),
    "platform": platform.platform(),
    "kernel": sys.executable,
    "package": str(installed),
    "verified_runtime_files": len(runtime_files),
    "runtime_sha256": runtime_digest.hexdigest(),
    "runtime_input_diff_sha256": hashlib.sha256(candidate_diff).hexdigest(),
    "uv_lock_sha256": hashlib.sha256((root / "uv.lock").read_bytes()).hexdigest(),
    "catalog_file_sha256": hashlib.sha256(
        (root / "configs/reference/binance-spot.json").read_bytes()
    ).hexdigest(),
    "notebook_source_sha256": notebook_digest,
    "experiment_inputs_sha256": input_digest,
    "notebook_tooling": {
        item.metadata["Name"]: item.version
        for item in distributions()
        if item.metadata["Name"] in {"nbclient", "nbformat", "ipykernel"}
    },
}
NotebookDisplay.show_table(("Identity", "Observed value"), list(identity.items()))

Identity,Observed value
notebook_path,/Users/lxg/Documents/GitHub/Alpha Research/alpha-research-lab-private/examples/research-foundations.ipynb
attempt_id,abc10b50-4ba5-4c67-82c8-fba30d9808b4
commit,dbd94b96abb545efd429dd2eb69bce1818b249f4
branch,codex/mr-07-acquisition-cli
working_tree,changed
python,3.14.7
platform,macOS-27.0-arm64-arm-64bit-Mach-O
kernel,/Users/lxg/.cache/uv/builds-v0/.tmpyC1CQ5/bin/python
package,/Users/lxg/Documents/GitHub/Alpha Research/alpha-research-lab-private/.venv/lib/python3.14/site-packages/alpha_research
verified_runtime_files,40


## 4. Resolve the catalog and compare decoded scenarios

Read the authored catalog and select the canonical instrument explicitly. Its
validity bounds are engineering metadata, not historical venue eligibility evidence.
Each scenario below calls the existing decoder; there is no notebook implementation
of timestamp parsing or money arithmetic.

In [4]:
domain_codec = DomainCodec()
catalog = domain_codec.decode_json((root / "configs/reference/binance-spot.json").read_bytes(), CatalogSnapshot)
instrument = catalog.resolve(
    InstrumentId.from_canonical_text(INSTRUMENT_ID), SAMPLE_START, SAMPLE_END
)
decoder = TardisDecoder(
    payload_decoder=BinancePayloadDecoder((instrument,)),
    source_id="tardis.binance.spot",
)
NotebookDisplay.show_table(
    ("Reference field", "Value"),
    [
        ("Instrument", instrument.symbol),
        ("Canonical ID", instrument.instrument_id),
        ("Currency", instrument.quote_currency),
        ("Tick", instrument.price_tick),
        ("Valid interval", f"[{instrument.valid_from}, {instrument.valid_to})"),
        ("Catalog identity", catalog.identity),
        ("Evidence", catalog.evidence),
    ],
)

events: dict[str, Trade] = {}
scenario_outcomes: list[tuple[object, ...]] = []
for sequence, scenario in enumerate(SCENARIOS):
    if scenario["name"] in {row[0] for row in scenario_outcomes}:
        raise ValueError("Scenario names must be unique")
    payload = {
        "stream": f"{scenario['symbol'].lower()}@trade",
        "data": {
            "e": "trade",
            "E": EVENT_TIME_MS + 1,
            "s": scenario["symbol"],
            "t": 7 + sequence,
            "p": scenario["price"],
            "q": scenario["quantity"],
            "T": EVENT_TIME_MS,
            "m": False,
        },
    }
    line = scenario["capture"] + " " + json.dumps(payload, separators=(",", ":"))
    try:
        event = decoder.decode_line(
            line,
            source_sequence=sequence,
            local_received=LOCAL_RECEIVED,
            local_ingested=LOCAL_INGESTED,
        )
    except MarketEventError as error:
        scenario_outcomes.append((scenario["name"], "rejected", str(error)))
    else:
        assert isinstance(event, Trade)
        events[scenario["name"]] = event
        scenario_outcomes.append((scenario["name"], "decoded", event.provenance.event_id))
NotebookDisplay.show_table(("Scenario", "Outcome", "Identity / reason"), scenario_outcomes)

dimension = Dimension(instrument_id=instrument.instrument_id, kind=QuantityKind.BASE_UNIT)
notionals: dict[str, Money] = {}
decoded_rows: list[tuple[object, ...]] = []
for name, event in events.items():
    notional = Price(value=event.price, currency=instrument.quote_currency, per=dimension).amount(
        Quantity(value=event.quantity, dimension=dimension)
    )
    notionals[name] = notional
    decoded_rows.append(
        (
            name,
            event.original_capture_time,
            event.provenance.ts_knowledge,
            event.price,
            event.quantity,
            notional.value,
            notional.currency.code,
        )
    )
NotebookDisplay.show_table(
    (
        "Scenario",
        "Source capture",
        "Knowledge UTC",
        "Price",
        "Quantity",
        "Exact notional",
        "Currency",
    ),
    decoded_rows,
)

Reference field,Value
Instrument,BTCUSDT
Canonical ID,11111111-1111-4111-8111-111111111111
Currency,code='USDT' minor_unit=Decimal('1E-8')
Tick,0.01
Valid interval,"[2024-06-01 00:00:00+00:00, 2024-06-02 00:00:00+00:00)"
Catalog identity,3125723d3760b4a35238b75a493e1d1e2a6956c66566ac7231c35a0afb59bad6
Evidence,Authored local research assumptions for the bounded Tardis Binance spot sample; this catalog does not assert historical exchange filters or point-in-time venue truth.


Scenario,Outcome,Identity / reason
reference,decoded,tardis.binance.spot:0:trade:7
exact_microsecond,decoded,tardis.binance.spot:1:trade:8
smaller_quantity,decoded,tardis.binance.spot:2:trade:9
unconfigured_symbol,rejected,unsupported stream symbol SOLUSDT


Scenario,Source capture,Knowledge UTC,Price,Quantity,Exact notional,Currency
reference,2024-06-01T00:00:00.0041023Z,2024-06-01 00:00:00.004103+00:00,67540.01000000,0.00146000,98.608414600000000000,USDT
exact_microsecond,2024-06-01T00:00:00.0041020Z,2024-06-01 00:00:00.004102+00:00,67540.01000000,0.00146000,98.608414600000000000,USDT
smaller_quantity,2024-06-01T00:00:00.0041023Z,2024-06-01 00:00:00.004103+00:00,67540.01000000,0.00100000,67.540010000000000000,USDT


## 5. Inspect clocks and availability at your chosen cutoffs

Change `FOCUS_SCENARIO` and `CUTOFF_OFFSETS_US` in the inputs cell. The two predicates
below ask different questions: is this assertion usable for its original business
instant, and is it valid at the decision instant itself? Point events have a
one-microsecond validity interval; this API is not a last-observation selector.

In [5]:
focus = events[FOCUS_SCENARIO]
provenance = focus.provenance
NotebookDisplay.show_table(
    ("Clock", "Observed UTC instant"),
    [
        ("Source event", provenance.ts_event),
        ("Source received", provenance.ts_source_received),
        ("Knowledge", provenance.ts_knowledge),
        ("Local received", provenance.ts_received),
        ("Local ingested", provenance.ts_ingested),
        ("Valid from", provenance.valid_from),
        ("Valid to (exclusive)", provenance.valid_to),
    ],
)
availability_rows: list[tuple[object, ...]] = []
for offset in CUTOFF_OFFSETS_US:
    cutoff = provenance.ts_knowledge + timedelta(microseconds=offset)
    availability_rows.append(
        (
            offset,
            cutoff,
            provenance.is_available(cutoff, provenance.valid_from),
            provenance.is_available(cutoff, cutoff),
        )
    )
NotebookDisplay.show_table(
    ("Cutoff offset (us)", "Decision cutoff", "Valid at original point?", "Valid at decision?"),
    availability_rows,
)

Clock,Observed UTC instant
Source event,2024-05-31 23:59:59.993000+00:00
Source received,2024-06-01 00:00:00.004103+00:00
Knowledge,2024-06-01 00:00:00.004103+00:00
Local received,2026-09-11 00:00:00+00:00
Local ingested,2026-09-11 00:00:01+00:00
Valid from,2024-06-01 00:00:00.004103+00:00
Valid to (exclusive),2024-06-01 00:00:00.004104+00:00


Cutoff offset (us),Decision cutoff,Valid at original point?,Valid at decision?
-1,2024-06-01 00:00:00.004102+00:00,False,False
0,2024-06-01 00:00:00.004103+00:00,True,True
1,2024-06-01 00:00:00.004104+00:00,True,False
1000000,2024-06-01 00:00:01.004103+00:00,True,False


## 6. Boundary exploration

Inspect a blank-line disconnect, a request that exceeds catalog coverage, and a
quantity for the wrong instrument. These exercise existing package boundaries.
The next-capture proxy remains a separate source-reader responsibility; this
notebook supplies the explicit interval-end proxy.

In [6]:
disconnect = decoder.decode_line(
    "",
    source_sequence=len(SCENARIOS),
    local_received=LOCAL_RECEIVED,
    local_ingested=LOCAL_INGESTED,
    disconnect_capture_time=SAMPLE_END,
    disconnect_capture_reason="interval_end",
)
assert isinstance(disconnect, Disconnect)
NotebookDisplay.show_table(
    ("Disconnect field", "Value"),
    [
        ("Proxy reason", disconnect.capture_proxy_reason),
        ("Knowledge", disconnect.provenance.ts_knowledge),
        ("Event", disconnect.provenance.ts_event),
        ("Instrument", disconnect.instrument_id),
        ("Source sequence", disconnect.provenance.source_sequence),
    ],
)
boundary_outcomes: list[tuple[object, ...]] = []
try:
    catalog.resolve(
        instrument.instrument_id,
        instrument.valid_from,
        instrument.valid_to + timedelta(microseconds=1),
    )
except ValueError as error:
    boundary_outcomes.append(("Catalog end + 1 microsecond", type(error).__name__, str(error)))
else:
    raise AssertionError("Incomplete catalog coverage was unexpectedly accepted")
other_dimension = Dimension(
    instrument_id=InstrumentId(UUID("33333333-3333-4333-8333-333333333333")),
    kind=QuantityKind.BASE_UNIT,
)
try:
    Price(value=focus.price, currency=instrument.quote_currency, per=dimension).amount(
        Quantity(value=Decimal("1"), dimension=other_dimension)
    )
except UnitError as error:
    boundary_outcomes.append(("Wrong quantity instrument", type(error).__name__, str(error)))
else:
    raise AssertionError("Mismatched quantity units were unexpectedly accepted")
NotebookDisplay.show_table(("Boundary", "Observed exception", "Reason"), boundary_outcomes)

Disconnect field,Value
Proxy reason,interval_end
Knowledge,2024-06-01 01:00:00+00:00
Event,2024-06-01 01:00:00+00:00
Instrument,None
Source sequence,4


Boundary,Observed exception,Reason
Catalog end + 1 microsecond,ValueError,instrument does not cover the complete requested interval
Wrong quantity instrument,UnitError,price and quantity dimension mismatch


## 7. Fixed reference examples — independent of the editable scenarios

Keep these constants fixed. They retain the original MR 00 clock, notional and
hand-ledger checks. Add alternate economics to a new exploratory section rather
than changing the expected golden result. These are reference calculations, not
a packaged fee engine or account ledger.

In [7]:
reference_start = datetime(2024, 6, 1, tzinfo=UTC)
reference_instrument = catalog.resolve(
    InstrumentId.from_canonical_text("11111111-1111-4111-8111-111111111111"),
    reference_start,
    reference_start + timedelta(hours=1),
)
reference_line = (
    "2024-06-01T00:00:00.0041023Z "
    '{"stream":"btcusdt@trade","data":{"e":"trade","E":1717199999994,'
    '"s":"BTCUSDT","t":7,"p":"67540.01000000","q":"0.00146000",'
    '"T":1717199999993,"m":false}}'
)
reference_received = datetime(2026, 9, 11, tzinfo=UTC)
reference_decoder = TardisDecoder(
    payload_decoder=BinancePayloadDecoder((reference_instrument,)),
    source_id="tardis.binance.spot",
)
reference_event = reference_decoder.decode_line(
    reference_line,
    source_sequence=0,
    local_received=reference_received,
    local_ingested=reference_received + timedelta(seconds=1),
    source_digest="a" * 64,
)
assert isinstance(reference_event, Trade)
reference_capture = reference_start + timedelta(microseconds=4103)
reference_provenance = reference_event.provenance
reference_dimension = Dimension(
    instrument_id=reference_instrument.instrument_id, kind=QuantityKind.BASE_UNIT
)
reference_notional = Price(
    value=reference_event.price,
    currency=reference_instrument.quote_currency,
    per=reference_dimension,
).amount(Quantity(value=reference_event.quantity, dimension=reference_dimension))
reference_checks = [
    ("Exchange clock", reference_provenance.ts_event, reference_start - timedelta(milliseconds=7)),
    ("Capture ceiling", reference_provenance.ts_source_received, reference_capture),
    ("Knowledge clock", reference_provenance.ts_knowledge, reference_capture),
    ("Local receipt", reference_provenance.ts_received, reference_received),
    (
        "Local ingestion",
        reference_provenance.ts_ingested,
        reference_received + timedelta(seconds=1),
    ),
    (
        "Point endpoint",
        reference_provenance.valid_to,
        reference_capture + timedelta(microseconds=1),
    ),
    ("Exact notional", reference_notional.value, Decimal("98.60841460")),
    ("USDT quantum", reference_notional.currency.minor_unit, Decimal("0.00000001")),
]
NotebookDisplay.show_table(
    ("Reference check", "Actual", "Expected", "Pass"),
    [(name, actual, expected, actual == expected) for name, actual, expected in reference_checks],
)
assert all(actual == expected for _, actual, expected in reference_checks)
assert reference_provenance.is_available(reference_capture, reference_capture)
assert not reference_provenance.is_available(
    reference_capture - timedelta(microseconds=1), reference_capture
)
assert not reference_provenance.is_available(
    reference_capture + timedelta(hours=1), reference_capture + timedelta(hours=1)
)

Reference check,Actual,Expected,Pass
Exchange clock,2024-05-31 23:59:59.993000+00:00,2024-05-31 23:59:59.993000+00:00,True
Capture ceiling,2024-06-01 00:00:00.004103+00:00,2024-06-01 00:00:00.004103+00:00,True
Knowledge clock,2024-06-01 00:00:00.004103+00:00,2024-06-01 00:00:00.004103+00:00,True
Local receipt,2026-09-11 00:00:00+00:00,2026-09-11 00:00:00+00:00,True
Local ingestion,2026-09-11 00:00:01+00:00,2026-09-11 00:00:01+00:00,True
Point endpoint,2024-06-01 00:00:00.004104+00:00,2024-06-01 00:00:00.004104+00:00,True
Exact notional,98.608414600000000000,98.60841460,True
USDT quantum,1E-8,1E-8,True


### Independent hand ledger

The arbitrary QXX currency has quantum `0.01`; fees use explicit Decimal half-up
rounding. This preserves the roadmap oracle. The selected P1 baseline instead
uses USDT quantum `0.00000001` and C03 half-even rounding. Neither choice here
claims a venue fee schedule. The reference instrument is deliberately arbitrary.

In [8]:
oracle_currency = Currency(code="QXX", minor_unit=Decimal("0.01"))
oracle_dimension = Dimension(
    instrument_id=InstrumentId(UUID("33333333-3333-4333-8333-333333333333")),
    kind=QuantityKind.BASE_UNIT,
)
oracle_quantity = Quantity(value=Decimal("2"), dimension=oracle_dimension)
buy = Price(value=Decimal("100.50"), currency=oracle_currency, per=oracle_dimension).amount(
    oracle_quantity
)
sell = Price(value=Decimal("109.50"), currency=oracle_currency, per=oracle_dimension).amount(
    oracle_quantity
)
buy_fee = Money(
    value=(buy.value * Decimal("0.001")).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP),
    currency=oracle_currency,
)
sell_fee = Money(
    value=(sell.value * Decimal("0.001")).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP),
    currency=oracle_currency,
)
initial = Money(value=Decimal("1000.00"), currency=oracle_currency)
cash_after_buy = initial - buy - buy_fee
open_mark = Price(value=Decimal("105.00"), currency=oracle_currency, per=oracle_dimension).amount(
    oracle_quantity
)
final_cash = cash_after_buy + sell - sell_fee
oracle_checks = [
    ("Buy fee", buy_fee.value, Decimal("0.20")),
    ("Sell fee", sell_fee.value, Decimal("0.22")),
    ("Cash after buy", cash_after_buy.value, Decimal("798.80")),
    ("Open NAV", (cash_after_buy + open_mark).value, Decimal("1008.80")),
    ("Final cash", final_cash.value, Decimal("1017.58")),
    ("Net P&L", (final_cash - initial).value, Decimal("17.58")),
]
NotebookDisplay.show_table(
    ("Golden", "Actual", "Expected", "Pass"),
    [(name, actual, expected, actual == expected) for name, actual, expected in oracle_checks],
)
assert all(actual == expected for _, actual, expected in oracle_checks)
assert (final_cash - initial).value == Decimal("20.00") - Decimal("2.00") - Decimal("0.42")
NotebookDisplay.show_table(
    ("Ledger step", "Cash change", "Cash", "Position", "NAV"),
    [
        ("Initial / no-trade", Decimal("0"), initial.value, Decimal("0"), initial.value),
        (
            "Buy; mark at 105",
            -(buy + buy_fee).value,
            cash_after_buy.value,
            Decimal("2"),
            (cash_after_buy + open_mark).value,
        ),
        ("Sell", (sell - sell_fee).value, final_cash.value, Decimal("0"), final_cash.value),
    ],
)

Golden,Actual,Expected,Pass
Buy fee,0.20,0.20,True
Sell fee,0.22,0.22,True
Cash after buy,798.800000000000000000,798.80,True
Open NAV,1008.800000000000000000,1008.80,True
Final cash,1017.580000000000000000,1017.58,True
Net P&L,17.580000000000000000,17.58,True


Ledger step,Cash change,Cash,Position,NAV
Initial / no-trade,0,1000.00,0,1000.00
Buy; mark at 105,-201.200000000000000000,798.800000000000000000,2,1008.800000000000000000
Sell,218.780000000000000000,1017.580000000000000000,0,1017.580000000000000000


## 8. Findings and next experiment

The computed summary describes this execution. Edit the interpretation and next
question below as ideas develop. Keep a new experiment ID when changing the
question or scenario set. Run through the runner to preserve a distinct executed
snapshot before replacing conclusions; direct interactive edits are not archived
automatically. Formal research still needs the retained trial fields and protected
input/output references; ignored local snapshots are not a durable research service.

In [9]:
findings = [
    ("Scenario results", f"{len(events)} decoded; {len(SCENARIOS) - len(events)} rejected"),
    ("Focused scenario", FOCUS_SCENARIO),
    ("Focused notional", f"{notionals[FOCUS_SCENARIO].value} {instrument.quote_currency.code}"),
    ("Observed rejection boundaries", len(boundary_outcomes)),
    ("Fixed clock/unit references", f"{len(reference_checks)} passed"),
    (
        "Fixed ledger references",
        f"{len(oracle_checks)} passed; net {(final_cash - initial).value} QXX",
    ),
    (
        "Current external evidence",
        "Synthetic Tardis foundation; actual P1 archive acquisition is in section 16",
    ),
]
NotebookDisplay.show_table(("Finding", "Evidence in this execution"), findings)

INTERPRETATION = (
    "Exact arithmetic and explicit temporal boundaries support controlled synthetic exploration."
)
NEXT_QUESTION = (
    "Which additional input boundary would improve understanding "
    "before implementing bounded source requests?"
)
PROMOTION_DECISION = (
    "Reuse the existing decoder, provenance and financial APIs. "
    "No new runtime implementation is needed for this experiment."
)
NotebookDisplay.show_table(
    ("Research note", "Current decision"),
    [
        ("Interpretation", INTERPRETATION),
        ("Next question", NEXT_QUESTION),
        ("Promotion", PROMOTION_DECISION),
    ],
)

Finding,Evidence in this execution
Scenario results,3 decoded; 1 rejected
Focused scenario,reference
Focused notional,98.608414600000000000 USDT
Observed rejection boundaries,2
Fixed clock/unit references,8 passed
Fixed ledger references,6 passed; net 17.580000000000000000 QXX
Current external evidence,Synthetic Tardis foundation; actual P1 archive acquisition is in section 16


Research note,Current decision
Interpretation,Exact arithmetic and explicit temporal boundaries support controlled synthetic exploration.
Next question,Which additional input boundary would improve understanding before implementing bounded source requests?
Promotion,"Reuse the existing decoder, provenance and financial APIs. No new runtime implementation is needed for this experiment."


## 9. Where this work goes next

| Working content | Eventual notebook | Promotion boundary |
| --- | --- | --- |
| Permitted requests, source messages and acquisition inspection | Data fetching | Stable acquisition, validation and storage APIs |
| Saved sample inspection, coverage, distributions and features | EDA | Shared transforms with retained consumers |
| Signals, economics, strategy comparisons and attribution | Strategy research | Selected versioned strategy and simulation/accounting implementations |
| Versioned reports and actual runtime observations | Strategy monitoring | Persistent metrics, alerts and operational controls |

These notebooks will emerge as working inputs and capabilities exist. The fixed
checks here remain backed by tracked tests; they do not require every exploratory
idea to become production code. Package APIs should own promoted behavior, and
notebooks should then call those implementations for formal results.

## 10. M0 configured saved-input inspection (NB 01-02)

Question: can a named universe select either supported spot instrument through the
same installed implementation, with exact minute partitions and explicit failures?
These are authored synthetic frames. The frozen P1 universe and source-access
status are unchanged. Configuration selects only fixture inspection.

The following cells consume the installed configuration, request and pipeline APIs;
they do not recreate decoders or request partitioning. Inspect both single-instrument
jobs and a two-instrument job, then run the installed command on saved bytes.


In [10]:
# third party library imports
from msgspec.structs import replace

# project library imports
from alpha_research.configuration import Configuration
from alpha_research.data.requests import SourceRequest
from alpha_research.pipelines.dataset import inspect_fixture

m0_config_path = root / "configs/data-jobs/fixture-inspection.json"
m0_fixture_root = root / "tests/fixtures/m0"
m0_configuration = Configuration.read(m0_config_path)
m0_rows = []
m0_manifests = {}
for job_name in ("btc-fixture", "eth-fixture", "both-fixture"):
    manifest = m0_configuration.resolve(job_name, catalog)
    requests = SourceRequest.partition(manifest)
    m0_manifests[job_name] = manifest
    assert len(requests) == 60 * len(manifest.instruments)
    assert requests[0].start == manifest.job.start
    assert requests[-1].end == manifest.job.end
    m0_rows.append((job_name, ", ".join(i.symbol for i in manifest.instruments), len(requests),
                    requests[0].start, requests[-1].end, requests[-1].offset_minutes,
                    manifest.identity))
NotebookDisplay.show_table(("Job", "Selected symbols", "Requests", "First start UTC", "Last end UTC",
            "Last minute offset", "Run identity"), m0_rows)
NotebookDisplay.show_table(("Offset", "Symbol", "Start UTC", "End UTC", "Request identity"),
           [(r.offset_minutes, r.instrument.symbol, r.start, r.end, r.identity)
            for r in (*requests[:4], *requests[-2:])])

m0_event_rows = []
for job_name, symbol in (("btc-fixture", "btcusdt"), ("eth-fixture", "ethusdt")):
    request, event, digest = inspect_fixture(
        m0_manifests[job_name], m0_fixture_root / f"{symbol}.txt",
        local_received=datetime(2026, 9, 14, tzinfo=UTC),
        local_ingested=datetime(2026, 9, 14, 0, 0, 1, tzinfo=UTC), source_sequence=0)
    assert event.instrument_id == request.instrument.instrument_id
    m0_event_rows.append((job_name, event.symbol, event.price, event.provenance.ts_event,
                          event.provenance.ts_knowledge, event.provenance.ts_received, digest))
NotebookDisplay.show_table(("Job", "Event symbol", "Exact price", "Source event UTC", "Capture knowledge UTC",
            "Original receipt UTC", "Fixture SHA-256"), m0_event_rows)


Job,Selected symbols,Requests,First start UTC,Last end UTC,Last minute offset,Run identity
btc-fixture,BTCUSDT,60,2024-06-01 00:00:00+00:00,2024-06-01 01:00:00+00:00,59,dfd73bc85524d2f9f5c73b6eda1d2dd1344c6ac5f75a05ed4d05600587816899
eth-fixture,ETHUSDT,60,2024-06-01 00:00:00+00:00,2024-06-01 01:00:00+00:00,59,4e8e9ee25db678666dec51a78684fdb2369d3b46aa720b343252283161e7961e
both-fixture,"BTCUSDT, ETHUSDT",120,2024-06-01 00:00:00+00:00,2024-06-01 01:00:00+00:00,59,1e47c2ab8991edd381bda9e5a890a014bed334f24ebcd52aa178319885a16ded


Offset,Symbol,Start UTC,End UTC,Request identity
0,BTCUSDT,2024-06-01 00:00:00+00:00,2024-06-01 00:01:00+00:00,d6b72573b8e41a4e791498e3bdc702054250e17fbeff218dbdaf85e4b82b57d1
0,ETHUSDT,2024-06-01 00:00:00+00:00,2024-06-01 00:01:00+00:00,e47ad10514e5de5ee8666cd1d5a705b2021e9f9fabc8e6bf7a9d0abce760d088
1,BTCUSDT,2024-06-01 00:01:00+00:00,2024-06-01 00:02:00+00:00,0c650711bb4e064982753f32ad6e23f7c5116e99af1d30b828aa8d706e13da45
1,ETHUSDT,2024-06-01 00:01:00+00:00,2024-06-01 00:02:00+00:00,19041592155589418b4398dee48f5e20d64185052e3b721feb129f6175bebd5b
59,BTCUSDT,2024-06-01 00:59:00+00:00,2024-06-01 01:00:00+00:00,c3c5dafcf54e7f5c15a7d83d713001a0c30b74db9d57bae05695072f372d4990
59,ETHUSDT,2024-06-01 00:59:00+00:00,2024-06-01 01:00:00+00:00,64165385576acac06727bfda17d818bf5d3984f67c9b0d0f9635ee5b2854d4ab


Job,Event symbol,Exact price,Source event UTC,Capture knowledge UTC,Original receipt UTC,Fixture SHA-256
btc-fixture,BTCUSDT,67540.01000000,2024-05-31 23:59:59.993000+00:00,2024-06-01 00:00:00.004103+00:00,2026-09-14 00:00:00+00:00,30a770208c525b39a5a9f8e9eb4f21b89bdb4b48bf063d6007a9174ed1b75915
eth-fixture,ETHUSDT,3800.01000000,2024-05-31 23:59:59.993000+00:00,2024-06-01 00:00:00.004103+00:00,2026-09-14 00:00:00+00:00,af186c0065aab0a4a841ef9cf447bb0f7ba2628fb6428b1367a4ebba8d7fbf87


In [11]:
m0_failures = []
try:
    domain_codec.decode_json(m0_config_path.read_text().replace('"product": "spot"',
                                                              '"product": "future"', 1), Configuration)
except ValueError as error:
    m0_failures.append(("Unsupported family", str(error)))
else:
    raise AssertionError("unsupported product was accepted")
try:
    shortened = replace(catalog, instruments=tuple(
        replace(i, valid_to=i.valid_from + timedelta(minutes=59)) for i in catalog.instruments))
    m0_configuration.resolve("btc-fixture", shortened)
except ValueError as error:
    m0_failures.append(("Incomplete catalog interval", str(error)))
else:
    raise AssertionError("incomplete catalog was accepted")
NotebookDisplay.show_table(("Rejected scenario", "Diagnostic"), m0_failures)

# standard library imports
import shutil

command_path = shutil.which("alpha-inspect-fixture")
assert command_path is not None, "Refresh the installed fixture command"
command = Path(command_path)
cli_rows = []
for job_name, filename, expected in (("btc-fixture", "btcusdt.txt", 0),
                                    ("eth-fixture", "ethusdt.txt", 0),
                                    ("missing-binding", "btcusdt.txt", 2),
                                    ("btc-fixture", "ethusdt.txt", 3),
                                    ("btc-fixture", "invalid-timestamp.txt", 3)):
    completed = subprocess.run(
        [str(command), "--config", str(m0_config_path), "--catalog",
         str(root / "configs/reference/binance-spot.json"), "--job", job_name,
         "--fixture", str(m0_fixture_root / filename), "--received", "2026-09-14T00:00:00Z",
         "--ingested", "2026-09-14T00:00:01Z", "--sequence", "0"],
        capture_output=True, text=True, check=False)
    assert completed.returncode == expected, completed.stderr
    report = json.loads(completed.stdout if expected == 0 else completed.stderr)
    cli_rows.append((job_name, filename, expected,
                    report["event"]["symbol"] if expected == 0 else report["diagnostic"],
                    report["run_identity"]))
NotebookDisplay.show_table(("Job", "Saved frame", "Exit", "Result/diagnostic", "Run identity"), cli_rows)
NotebookDisplay.show_table(("M0 input", "SHA-256"),
           [(str(path.relative_to(root)), hashlib.sha256(path.read_bytes()).hexdigest())
            for path in (m0_config_path, *sorted(m0_fixture_root.iterdir()))])


Rejected scenario,Diagnostic
Unsupported family,Invalid enum value 'future' - at `$.jobs[0].product`
Incomplete catalog interval,instrument does not cover the complete requested interval


Job,Saved frame,Exit,Result/diagnostic,Run identity
btc-fixture,btcusdt.txt,0,BTCUSDT,dfd73bc85524d2f9f5c73b6eda1d2dd1344c6ac5f75a05ed4d05600587816899
eth-fixture,ethusdt.txt,0,ETHUSDT,4e8e9ee25db678666dec51a78684fdb2369d3b46aa720b343252283161e7961e
missing-binding,btcusdt.txt,2,unknown data-job binding: missing-binding,None
btc-fixture,ethusdt.txt,3,unsupported stream symbol ETHUSDT,dfd73bc85524d2f9f5c73b6eda1d2dd1344c6ac5f75a05ed4d05600587816899
btc-fixture,invalid-timestamp.txt,3,T must be nonnegative,dfd73bc85524d2f9f5c73b6eda1d2dd1344c6ac5f75a05ed4d05600587816899


M0 input,SHA-256
configs/data-jobs/fixture-inspection.json,cbbda41513b34530118982194372a1ea09e22d1800a37e7c654a6b3872b0b3ab
tests/fixtures/m0/btcusdt.txt,30a770208c525b39a5a9f8e9eb4f21b89bdb4b48bf063d6007a9174ed1b75915
tests/fixtures/m0/ethusdt.txt,af186c0065aab0a4a841ef9cf447bb0f7ba2628fb6428b1367a4ebba8d7fbf87
tests/fixtures/m0/invalid-timestamp.txt,59977c0f5ce69c851ac0e680281b65f20df4ad18d167c2ccd05c67fbc423e53b
tests/fixtures/m0/unsupported-product.json,3ec687324a567a041dd14364185ef458354d018c803161bedcf8dc1e70584b40


### M0 findings

Both supported listings and the combined universe resolve through the same installed
code. Each selected hour yields exactly 60 minute requests per instrument. The
saved frames preserve exact prices, the preceding exchange event time and the
conservatively rounded capture time; capture time owns partition membership.
Unknown bindings, an unsupported family, incomplete catalog coverage, wrong
instruments and invalid exchange timestamps fail explicitly.

This is fixture evidence, with no provider access, historical eligibility proof,
raw-object store, dataset publication, or strategy execution. Section 11 now exercises MR 03's saved raw-object descriptor/verifier. Keep MR 05's permitted
source feasibility work early. The frozen experiment and its hand ledger are unchanged.


## 11. Describe and verify retained raw bytes (NB 03)

Question: does the saved object still contain the exact bytes assigned to this
request partition? The installed `RawObject` records SHA-256, byte size, the
existing request identity, a synthetic retrieval reference and original local
receipt/ingestion times. It does not infer source validity or replace the clocks
with verification time. The factory and verifier read in bounded binary chunks.

The sample below compresses the retained synthetic BTC frame, verifies and
round-trips its descriptor, then changes one gzip header byte. Decompression still
yields the same frame, but byte identity must fail. Truncation and a descriptor
used for another minute must also fail. All files live in a temporary directory.


In [12]:
# standard library imports
import gzip
from tempfile import TemporaryDirectory

# project library imports
from alpha_research.data.raw_objects import RawObject

raw_request = SourceRequest.partition(m0_manifests["btc-fixture"])[0]
raw_payload = (m0_fixture_root / "btcusdt.txt").read_bytes()
raw_compressed = gzip.compress(raw_payload, mtime=0)
raw_failures = []
with TemporaryDirectory(prefix="mr03-notebook-") as raw_directory:
    raw_path = Path(raw_directory) / "fixture.gz"
    raw_path.write_bytes(raw_compressed)
    raw_descriptor = RawObject.from_file(
        raw_path,
        request=raw_request,
        retrieval_reference="synthetic:m0-btc-gzip-v1",
        ts_received=datetime(2026, 9, 14, tzinfo=UTC),
        ts_ingested=datetime(2026, 9, 14, 0, 0, 1, tzinfo=UTC),
    )
    raw_wire = domain_codec.encode_json(raw_descriptor)
    raw_restored = domain_codec.decode_json(raw_wire, RawObject)
    assert raw_restored == raw_descriptor
    assert raw_restored.verify(raw_path, request=raw_request) is raw_restored
    assert raw_descriptor.sha256 == hashlib.sha256(raw_compressed).hexdigest()
    assert raw_descriptor.sha256 != hashlib.sha256(raw_payload).hexdigest()
    NotebookDisplay.show_table(
        ("Raw object field", "Verified value"),
        [
            ("Retained size (compressed bytes)", raw_descriptor.size_bytes),
            ("Retained SHA-256", raw_descriptor.sha256),
            ("Decompressed SHA-256 (different identity)", hashlib.sha256(raw_payload).hexdigest()),
            ("Request / partition identity", raw_descriptor.request_identity),
            ("Partition bounds [start, end)", f"{raw_request.start} / {raw_request.end}"),
            ("Retrieval reference", raw_descriptor.retrieval_reference),
            ("Original local receipt UTC", raw_descriptor.ts_received),
            ("Original local ingestion UTC", raw_descriptor.ts_ingested),
            ("Descriptor JSON round trip", raw_restored == raw_descriptor),
        ],
    )
    changed_header = raw_compressed[:4] + b"\x01" + raw_compressed[5:]
    assert gzip.decompress(changed_header) == raw_payload
    for label, payload, expected_request, message in (
        (
            "One changed header byte; same decoded content",
            changed_header,
            raw_request,
            "SHA-256 mismatch",
        ),
        ("Truncated retained bytes", raw_compressed[:-1], raw_request, "size mismatch"),
        (
            "Descriptor for another minute",
            raw_compressed,
            replace(raw_request, offset_minutes=1),
            "request identity mismatch",
        ),
    ):
        raw_path.write_bytes(payload)
        try:
            raw_restored.verify(raw_path, request=expected_request)
        except ValueError as error:
            assert message in str(error)
            raw_failures.append((label, str(error)))
        else:
            raise AssertionError(f"Accepted invalid raw object: {label}")
    raw_path.write_bytes(raw_compressed)
    assert raw_restored.verify(raw_path, request=raw_request) is raw_restored
    assert domain_codec.encode_json(raw_restored) == raw_wire
NotebookDisplay.show_table(("Rejected scenario", "Diagnostic"), raw_failures)
print(
    "NB 03: verified compressed bytes, retained clocks and JSON round trip; 3 expected failures; recovery passed."
)


Raw object field,Verified value
Retained size (compressed bytes),145
Retained SHA-256,e4a81e816955ccd1884b30986a6fea04c10b8c0eafd14233d0341cc6a0b19d26
Decompressed SHA-256 (different identity),30a770208c525b39a5a9f8e9eb4f21b89bdb4b48bf063d6007a9174ed1b75915
Request / partition identity,d6b72573b8e41a4e791498e3bdc702054250e17fbeff218dbdaf85e4b82b57d1
"Partition bounds [start, end)",2024-06-01 00:00:00+00:00 / 2024-06-01 00:01:00+00:00
Retrieval reference,synthetic:m0-btc-gzip-v1
Original local receipt UTC,2026-09-14 00:00:00+00:00
Original local ingestion UTC,2026-09-14 00:00:01+00:00
Descriptor JSON round trip,True


Rejected scenario,Diagnostic
One changed header byte; same decoded content,"raw object SHA-256 mismatch: expected e4a81e816955ccd1884b30986a6fea04c10b8c0eafd14233d0341cc6a0b19d26, got ecf46d7bdffc36f0101ac37041fe603c7063df5a494eac60ab9e0b30e8b038cd"
Truncated retained bytes,"raw object size mismatch: expected 145, got 144"
Descriptor for another minute,raw object request identity mismatch


NB 03: verified compressed bytes, retained clocks and JSON round trip; 3 expected failures; recovery passed.


### MR 03 findings and limits

Exact stored bytes own identity: even a gzip header change with identical decoded
content is rejected. A shorter file fails its byte count; the same bytes cannot
satisfy a descriptor assigned to a different request minute. Reopening after the
failures succeeds with the original bytes, descriptor and timestamps.

This verifies an existing local object against trusted metadata at read time.
The factory records the caller's attribution; it cannot authenticate a provider
or establish response completeness. Concurrent writers must be excluded by the
caller. No raw publication, acquisition, resume or dataset validity is claimed.
Section 12 exercises MR 04 publication separately. The frozen P1 experiment
remains unchanged.


## 12. Durable raw publication and interrupted writers (NB 04)

Question: what can a reader observe when a publisher dies immediately before or
after atomic publication? `RawStore` accepts an existing storage directory and
stages on that filesystem. It verifies exact bytes, synchronizes the file, links
the digest name without overwriting a winner, then synchronizes the directory.
`open` checks the complete published object before returning a read-only stream.

Two isolated child processes use the retained synthetic interruption fixture and
exit without cleanup at the actual link boundary. Inspect the leftover staging
file and published name, retry through a new store instance, and confirm the
published inode is reused. Remove an orphan only after its writer has exited.
These are process-failure experiments, not physical power-loss tests.


In [13]:
# project library imports
from alpha_research.data.raw_store import RawStore

publication_rows = []
with TemporaryDirectory(prefix="mr04-notebook-") as publication_directory:
    publication_root = Path(publication_directory)
    descriptor_path = publication_root / "descriptor.json"
    descriptor_path.write_bytes(raw_wire)
    request_path = publication_root / "request.json"
    request_path.write_bytes(domain_codec.encode_json(raw_request))
    for point in ("before", "after"):
        store_root = publication_root / point
        store_root.mkdir()
        source = publication_root / f"{point}.gz"
        source.write_bytes(raw_compressed)
        attempt = subprocess.run(
            [sys.executable, "-I", str(root / "tests/fixtures/mr04/interrupt_publish.py"),
             str(store_root), str(source), str(descriptor_path), str(request_path), point],
            cwd=publication_root, capture_output=True, text=True, check=False, timeout=30,
        )
        assert attempt.returncode == 75, attempt.stderr
        staging, = store_root.glob(".staging-*")
        raw_descriptor.verify(staging, request=raw_request)
        store = RawStore(store_root)
        published = store_root / raw_descriptor.sha256
        if point == "before":
            try:
                store.open(descriptor=raw_descriptor, request=raw_request)
            except FileNotFoundError:
                reader_state = "Missing: staging is invisible to readers"
            else:
                raise AssertionError("Reader accepted an unpublished object")
        else:
            with store.open(descriptor=raw_descriptor, request=raw_request) as stream:
                assert stream.read() == raw_compressed
            assert published.stat().st_ino == staging.stat().st_ino
            source.unlink()
            reader_state = f"Complete and verified: {raw_descriptor.size_bytes} bytes"
        publication_rows.append((f"Exit {point} link", "1 verified orphan", reader_state))
        recovered = store.publish(source, descriptor=raw_descriptor, request=raw_request)
        recovered_inode = recovered.stat().st_ino
        assert store.publish(source, descriptor=raw_descriptor, request=raw_request) == recovered
        assert recovered.stat().st_ino == recovered_inode
        with store.open(descriptor=raw_descriptor, request=raw_request) as stream:
            assert stream.read() == raw_compressed
        assert domain_codec.encode_json(raw_descriptor) == raw_wire
        staging.unlink()  # This child has exited; no active publisher owns the orphan.
        assert list(store_root.iterdir()) == [published]
        publication_rows.append((f"Retry {point} link", "0 after orphan cleanup", "Verified; repeat reuses inode"))
    published.chmod(0o644)  # Inject corruption by an external writer in this temporary experiment.
    published.write_bytes(b"X" + raw_compressed[1:])
    for operation in ("open", "publish"):
        try:
            if operation == "open":
                store.open(descriptor=raw_descriptor, request=raw_request)
            else:
                store.publish(source, descriptor=raw_descriptor, request=raw_request)
        except ValueError as error:
            assert "SHA-256 mismatch" in str(error)
            publication_rows.append((f"Corrupt existing object: {operation}", "0", "Rejected: SHA-256 mismatch"))
        else:
            raise AssertionError("Accepted a corrupt published object")
    assert published.read_bytes() == b"X" + raw_compressed[1:]
NotebookDisplay.show_table(("Boundary", "Staging state", "Reader / retry outcome"), publication_rows)
print("NB 04: 2 abrupt exits; 2 reconciled retries; 2 conflict rejections; original descriptor and clocks retained.")


Boundary,Staging state,Reader / retry outcome
Exit before link,1 verified orphan,Missing: staging is invisible to readers
Retry before link,0 after orphan cleanup,Verified; repeat reuses inode
Exit after link,1 verified orphan,Complete and verified: 145 bytes
Retry after link,0 after orphan cleanup,Verified; repeat reuses inode
Corrupt existing object: open,0,Rejected: SHA-256 mismatch
Corrupt existing object: publish,0,Rejected: SHA-256 mismatch


NB 04: 2 abrupt exits; 2 reconciled retries; 2 conflict rejections; original descriptor and clocks retained.


### MR 04 findings and limits

Before the link, only hidden staging exists and the reader reports a missing
object. After the link, the reader sees the complete verified compressed object,
while the interrupted call has not acknowledged durable completion. Retry either
publishes from the saved source or verifies the existing object and synchronizes
it again. Repetition preserves its inode; descriptors and original clocks remain
unchanged. Corrupt existing bytes fail both read and publish and are not replaced.

This boundary requires a caller-controlled, durably provisioned directory on a
local filesystem supporting hard links and file/directory synchronization. External
mutation/removal must be excluded during verified reads. Read-only file mode is
an accidental-write guard, not protection from the directory owner. Orphan staging
is ignored; remove it only with writers stopped. No acquisition, persistent
manifest, provider validity, dataset publication or empirical P1 result is claimed.
The next package is **MR 05 — Fetch one provider partition**; it is not started.


## 13. Fetch and inspect a saved partition (NB 05)

Use an injected HTTP response to fetch, validate gzip, publish and reopen the exact compressed bytes. Decode the saved trade with the retained Tardis and Binance decoders. The sample and clock values are synthetic. No network access occurs.

This demonstrates raw-byte acquisition; source permission, actual framing and historical timestamp units still need a permitted probe.

In [14]:
# standard library imports
import socket
from http.client import HTTPResponse, HTTPSConnection
from io import BufferedReader, BytesIO
from unittest.mock import patch

# project library imports
from alpha_research.data.providers.tardis.transport import TransientFetchError
from alpha_research.pipelines.acquisition import fetch_partition


class NotebookResponse:
    """Parse a synthetic wire response with the standard HTTP parser, without connecting."""

    @staticmethod
    def parse(payload: bytes, status: int = 200) -> HTTPResponse:
        """Prepare one gzip HTTP response for the injected transport boundary."""
        wire = (
            f"HTTP/1.1 {status} Synthetic\r\nContent-Encoding: gzip\r\n"
            f"Content-Length: {len(payload)}\r\n\r\n"
        ).encode() + payload
        with socket.socket() as sock:
            response = HTTPResponse(sock)
        response.fp.close()
        response.fp = BufferedReader(BytesIO(wire))
        response.begin()
        return response


acquisition_manifest = replace(
    m0_manifests["btc-fixture"],
    job=replace(m0_manifests["btc-fixture"].job, mode="historical", pipeline="acquisition.fetch"),
)
acquisition_request = SourceRequest.partition(acquisition_manifest)[0]
acquisition_payload = gzip.compress((m0_fixture_root / "btcusdt.txt").read_bytes(), mtime=0)
acquisition_reports = []
with TemporaryDirectory(prefix="mr05-notebook-") as directory:
    acquisition_store = RawStore(Path(directory))
    acquisition_times = iter((datetime(2026, 9, 15, tzinfo=UTC), datetime(2026, 9, 15, 0, 0, 1, tzinfo=UTC)))
    with patch.object(HTTPSConnection, "request", return_value=None), patch.object(
        HTTPSConnection, "getresponse", return_value=NotebookResponse.parse(acquisition_payload)
    ):
        acquired = fetch_partition(
            acquisition_request, store=acquisition_store, connection=HTTPSConnection("api.tardis.dev"),
            clock=acquisition_times.__next__, report=acquisition_reports.append,
            timeout_seconds=1.0, max_bytes=4096,
            permitted_rights_reference="synthetic-local-fixtures-v1",
        )
    with acquisition_store.open(descriptor=acquired, request=acquisition_request) as saved:
        retained = saved.read()
    assert retained == acquisition_payload
    saved_line = gzip.decompress(retained).decode().removesuffix("\n")
    saved_decoder = TardisDecoder(BinancePayloadDecoder(acquisition_manifest.instruments),
                                 source_id=acquisition_request.instrument.source)
    saved_trade = saved_decoder.decode_line(
        saved_line, source_sequence=0, local_received=acquired.ts_received,
        local_ingested=acquired.ts_ingested, source_digest=acquired.sha256,
    )
    assert saved_trade.price == Decimal("67540.01000000")
    assert saved_trade.original_capture_time == "2024-06-01T00:00:00.0041023Z"
    assert saved_trade.provenance.ts_received == acquired.ts_received
    NotebookDisplay.show_table(("Acquisition fact", "Observed value"), [
        ("Run identity", acquisition_manifest.identity), ("Partition identity", acquired.request_identity),
        ("Retained gzip SHA-256", acquired.sha256), ("Compressed bytes", acquired.size_bytes),
        ("Source capture", saved_trade.original_capture_time),
        ("Exchange trade time", saved_trade.provenance.ts_event),
        ("Local receipt", acquired.ts_received), ("Durable ingestion", acquired.ts_ingested),
        ("Exact price / quantity", f"{saved_trade.price} / {saved_trade.quantity}"),
    ])
    progress_rows = [json.loads(row) for row in acquisition_reports]
    NotebookDisplay.show_table(("Stage", "Status", "Bytes"),
                              [(row["stage"], row["status"], row["bytes"]) for row in progress_rows])

Acquisition fact,Observed value
Run identity,e4235ad7263a8879b61ceb0e0a6fad76e0fee337fc3b4ce0f3dc2b27bdbccc3e
Partition identity,664d2cce7ddec3ccf818b8095f076573191ef8e6205c39666a88ec9ca9b8f808
Retained gzip SHA-256,e4a81e816955ccd1884b30986a6fea04c10b8c0eafd14233d0341cc6a0b19d26
Compressed bytes,145
Source capture,2024-06-01T00:00:00.0041023Z
Exchange trade time,2024-05-31 23:59:59.993000+00:00
Local receipt,2026-09-15 00:00:00+00:00
Durable ingestion,2026-09-15 00:00:01+00:00
Exact price / quantity,67540.01000000 / 0.00146000


Stage,Status,Bytes
fetch,started,0
store,received,145
store,stored,145


### Failure boundaries

Inject malformed/truncated gzip, an auth denial and a socket timeout. Each failure must emit a terminal status and leave storage empty.

In [15]:
acquisition_failures = []
for label, payload, status, expected in (
    ("Malformed gzip", b"not gzip", 200, ValueError),
    ("Truncated gzip", acquisition_payload[:-4], 200, ValueError),
    ("Authentication", b"synthetic denied", 401, PermissionError),
    ("Timeout", acquisition_payload, 200, TransientFetchError),
):
    with TemporaryDirectory(prefix="mr05-rejection-") as directory:
        failure_reports = []
        error = TimeoutError("injected socket timeout") if label == "Timeout" else None
        with patch.object(HTTPSConnection, "request", side_effect=error), patch.object(
            HTTPSConnection, "getresponse", return_value=NotebookResponse.parse(payload, status)
        ) as response_patch:
            try:
                fetch_partition(
                    acquisition_request, store=RawStore(Path(directory)), connection=HTTPSConnection("api.tardis.dev"),
                    clock=acquisition_times.__next__, report=failure_reports.append,
                    timeout_seconds=1.0, max_bytes=4096,
                    permitted_rights_reference="synthetic-local-fixtures-v1",
                )
            except expected as failure:
                assert list(Path(directory).iterdir()) == []
                terminal = json.loads(failure_reports[-1])
                assert terminal["status"] == "failed"
                acquisition_failures.append((label, type(failure).__name__, terminal["stage"], "No published bytes"))
            else:
                raise AssertionError(f"{label} unexpectedly succeeded")
            finally:
                response_patch.return_value.close()
NotebookDisplay.show_table(("Injected failure", "Exception", "Failed stage", "Storage"), acquisition_failures)


Injected failure,Exception,Failed stage,Storage
Malformed gzip,ValueError,fetch,No published bytes
Truncated gzip,ValueError,fetch,No published bytes
Authentication,PermissionError,fetch,No published bytes
Timeout,TransientFetchError,fetch,No published bytes


### MR 05 findings and limits

The saved response preserves the gzip digest and exact price, quantity, source capture, exchange event and original local clocks. Successful progress ends only after durable publication. Malformed/truncated gzip, authentication and timeout failures publish nothing and emit failure status.

The transport validates HTTP/gzip integrity; semantic frame validation, disconnect iteration, coverage and canonical dataset publication remain later work. The socket timeout bounds inactivity rather than total wall time. At MR 05, no Tardis source probe was performed because its rights admission and retention were unresolved. The founder later switched P1 to Binance archives; section 16 demonstrates actual acquisition. Historical eligibility and revision-availability claims remain unproven. See the [MR 05 record](../docs/roadmap/mr-05-implementation.md). Section 14 adds MR 06 resume; this section remains the one-partition checkpoint.

## 14. Resume a three-partition acquisition (NB 06)

Question: can a fresh process finish interrupted acquisition without losing or repeating logical observations?
The installed pipeline persists a receipt before raw publication and appends a completion afterward.
The retained [synthetic process fixture](../tests/fixtures/mr06/acquire.py) supplies three distinct gzip observations over fake HTTP. It exits abruptly at the second partition, before/after raw publication or checkpoint replacement; actual files, atomic operations and synchronization remain in use.

Compare the full resumed manifest with an uninterrupted run. Fixed UTC clocks make the documents byte-identical here. In an actual interrupted run, a pending receipt keeps its original receipt time and records ingestion when recovery reconfirms durability; completed clocks never change. No external source is contacted.


In [16]:
# project library imports
from alpha_research.data.manifests import AcquisitionManifest
from alpha_research.data.providers.tardis.transport import TardisTransport
from alpha_research.pipelines.acquisition import acquire


class NotebookResume:
    """Drive retained synthetic process fixtures and inspect their saved checkpoints."""

    @staticmethod
    def process(destination: Path, run_path: Path, point: str) -> subprocess.CompletedProcess[str]:
        """Start the candidate installed interpreter outside the checkout."""
        return subprocess.run(
            [sys.executable, "-I", str(root / "tests/fixtures/mr06/acquire.py"),
             str(destination), str(run_path), point],
            cwd=destination, capture_output=True, text=True, timeout=30, check=False,
        )

    @staticmethod
    def clock() -> datetime:
        """Provide the fixture's fixed UTC instant for deterministic comparisons."""
        return datetime(2026, 9, 15, tzinfo=UTC)


resume_run = replace(
    acquisition_manifest,
    job=replace(acquisition_manifest.job, end=acquisition_manifest.job.start + timedelta(minutes=3)),
)
resume_rows = []
resume_failures = []
with TemporaryDirectory(prefix="mr06-notebook-") as directory:
    resume_root = Path(directory)
    run_path = resume_root / "run.json"
    run_path.write_bytes(domain_codec.encode_json(resume_run))
    baseline = resume_root / "baseline"
    (baseline / "raw").mkdir(parents=True)
    result = NotebookResume.process(baseline, run_path, "none")
    assert result.returncode == 0, result.stderr
    expected_manifest = (baseline / "manifest.json").read_bytes()
    expected_state = domain_codec.decode_json(expected_manifest, AcquisitionManifest)
    assert len(expected_state.completed) == 3 and expected_state.pending is None
    for point in ("before-raw", "after-raw", "before-checkpoint", "after-checkpoint"):
        destination = resume_root / point
        (destination / "raw").mkdir(parents=True)
        result = NotebookResume.process(destination, run_path, point)
        assert result.returncode == 75, result.stderr
        checkpoint = domain_codec.decode_json((destination / "manifest.json").read_bytes(), AcquisitionManifest)
        raw_count = len([p for p in (destination / "raw").iterdir() if not p.name.startswith(".")])
        result = NotebookResume.process(destination, run_path, "none")
        assert result.returncode == 0, result.stderr
        assert (destination / "manifest.json").read_bytes() == expected_manifest
        fetches = (destination / "fetches.txt").read_text().splitlines()
        assert fetches == (["0", "1", "1", "2"] if point == "before-raw" else ["0", "1", "2"])
        result = NotebookResume.process(destination, run_path, "none")
        assert result.returncode == 0, result.stderr
        assert (destination / "fetches.txt").read_text().splitlines() == fetches
        resume_rows.append((point, len(checkpoint.completed), checkpoint.pending is not None,
                            raw_count, ", ".join(fetches), "3; exact manifest match; repeat skips all"))
    NotebookDisplay.show_table(
        ("Exit boundary", "Completed before", "Pending receipt", "Raw objects before", "Fetched offsets", "After resume"),
        resume_rows,
    )
    NotebookDisplay.show_table(("Offset", "Request identity", "Gzip digest", "Original receipt"), [
        (request.offset_minutes, descriptor.request_identity[:16], descriptor.sha256[:16], descriptor.ts_received)
        for request, descriptor in zip(SourceRequest.partition(resume_run), expected_state.completed)
    ])
    for label in ("missing", "mutated"):
        destination = resume_root / label
        (destination / "raw").mkdir(parents=True)
        (destination / "manifest.json").write_bytes(expected_manifest)
        for original in (baseline / "raw").iterdir():
            (destination / "raw" / original.name).write_bytes(original.read_bytes())
        damaged = destination / "raw" / expected_state.completed[0].sha256
        if label == "missing":
            damaged.unlink()
        else:
            payload = damaged.read_bytes()
            damaged.write_bytes(b"X" + payload[1:])
        result = NotebookResume.process(destination, run_path, "none")
        assert result.returncode != 0 and not (destination / "fetches.txt").exists()
        expected_error = "FileNotFoundError" if label == "missing" else "SHA-256 mismatch"
        assert expected_error in result.stderr
        assert (destination / "manifest.json").read_bytes() == expected_manifest
        resume_failures.append((f"Completed object {label}", expected_error, 0, "No source access; manifest retained"))
NotebookDisplay.show_table(("Failure", "Rejection", "Fetches", "Retained state"), resume_failures)
print("NB 06: four process interruptions recovered; three unique completions; completed bytes fail closed.")


Exit boundary,Completed before,Pending receipt,Raw objects before,Fetched offsets,After resume
before-raw,1,True,1,"0, 1, 1, 2",3; exact manifest match; repeat skips all
after-raw,1,True,2,"0, 1, 2",3; exact manifest match; repeat skips all
before-checkpoint,1,True,2,"0, 1, 2",3; exact manifest match; repeat skips all
after-checkpoint,2,False,2,"0, 1, 2",3; exact manifest match; repeat skips all


Offset,Request identity,Gzip digest,Original receipt
0,664d2cce7ddec3cc,7e0577039da97144,2026-09-15 00:00:00+00:00
1,a5ac5d7afacaa328,ca0da4d50e10da90,2026-09-15 00:00:00+00:00
2,a3e67203df6879ee,594da3490f761c20,2026-09-15 00:00:00+00:00


Failure,Rejection,Fetches,Retained state
Completed object missing,FileNotFoundError,0,No source access; manifest retained
Completed object mutated,SHA-256 mismatch,0,No source access; manifest retained


NB 06: four process interruptions recovered; three unique completions; completed bytes fail closed.


### Terminal failures and bounded backoff

A transient transport exception alone permits retry. The tuple `(0.1, 0.2)` allows two additional attempts; the injected sleep records these delays without waiting. Authentication and schema failures stop on their first attempt. Each failure leaves an incomplete saved manifest. Storage and checkpoint failures also stop the invocation; reopen the manifest before retrying an uncertain write.


In [17]:
retry_rows = []
for label, failure, expected_attempts in (
    ("Authentication", PermissionError("synthetic denial"), 1),
    ("Response schema", ValueError("synthetic invalid gzip"), 1),
    ("Exhausted transient", TransientFetchError("synthetic unavailable"), 3),
):
    with TemporaryDirectory(prefix="mr06-retry-") as directory:
        destination = Path(directory)
        (destination / "raw").mkdir()
        waits, reports = [], []
        with patch.object(TardisTransport, "fetch", side_effect=failure) as fetch:
            try:
                acquire(
                    resume_run, manifest_path=destination / "manifest.json", store=RawStore(destination / "raw"),
                    connection=HTTPSConnection("api.tardis.dev"), clock=NotebookResume.clock,
                    sleep=waits.append, report=reports.append, timeout_seconds=1.0, max_bytes=4096,
                    permitted_rights_reference="synthetic-local-fixtures-v1", retry_delays=(0.1, 0.2),
                )
            except type(failure):
                saved = AcquisitionManifest.read(destination / "manifest.json", run=resume_run, store=RawStore(destination / "raw"))
                assert not saved.completed and saved.pending is None
                assert fetch.call_count == expected_attempts
                assert waits == ([0.1, 0.2] if expected_attempts == 3 else [])
                retry_rows.append((label, fetch.call_count, str(waits), len(saved.completed), "Incomplete; exception propagated"))
            else:
                raise AssertionError(f"{label} unexpectedly completed")
NotebookDisplay.show_table(("Failure", "Attempts", "Backoff seconds", "Completed", "Run outcome"), retry_rows)


Failure,Attempts,Backoff seconds,Completed,Run outcome
Authentication,1,[],0,Incomplete; exception propagated
Response schema,1,[],0,Incomplete; exception propagated
Exhausted transient,3,"[0.1, 0.2]",0,Incomplete; exception propagated


### MR 06 findings and limits

Resume verifies the entire completed prefix before new source access. A receipt without a published object is fetched again; published pending bytes are verified and reused. Checkpoints preserve one completion per selected request, and repeated complete runs fetch nothing. Missing or modified completed objects fail instead of being replaced. Authentication/schema failures are terminal; exhausted transient attempts retain incomplete state.

The manifest uses one caller-controlled file with atomic replacement and file/directory synchronization. One writer owns the file; it rewrites and validates the bounded completion prefix at each checkpoint. Tests inject process death and I/O failure, not physical power loss. Hidden staging files from dead writers are ignored. Original receipt clocks survive recovery; only unacknowledged ingestion is timestamped when durability is confirmed again.

This is synthetic development evidence. The Tardis access boundary recorded in MR 05 applies to these historical synthetic fixtures. Section 16 uses the later founder-selected Binance public archive; historical eligibility and revision availability remain unproven. Section 15 exercises the installed acquisition command and expanded telemetry added in MR 07. Frame iteration/semantic validation belong to later MRs; the synthetic observation bytes here are not a provider-data acceptance claim. See the [MR 06 record](../docs/roadmap/mr-06-implementation.md).


## 15. Installed acquisition command and observable restart (NB 07)

Question: can a saved command/configuration resume from another directory and expose truthful progress?
The child process executes the installed `alpha-acquire` entry point with copied configuration and the existing synthetic HTTP/process fixture. Real filesystem persistence remains active; no endpoint is contacted.

Inspect the last acknowledged checkpoint before an abrupt exit, a keyboard interruption and retry exhaustion. Resume each command, then repeat it: logical completions, bytes and original completed clocks must remain unchanged. Credentials are never passed on the command line.


In [18]:
# standard library imports
import shutil
from importlib.metadata import distribution, entry_points

# project library imports
from alpha_research.pipelines.acquisition import main as acquisition_main

assert callable(acquisition_main)
assert next(iter(entry_points(group="console_scripts", name="alpha-acquire"))).value == "alpha_research.pipelines.acquisition:main"
command_distribution = distribution("alpha-research-lab")
command_script = Path(command_distribution.locate_file(next(
    item for item in command_distribution.files if item.name == "alpha-acquire"
))).resolve()
assert command_script.is_file()


class NotebookAcquisitionCommand:
    """Drive the installed console script with copied inputs and a synthetic HTTP boundary."""

    @staticmethod
    def process(destination: Path, arguments: list[str], point: str | None) -> subprocess.CompletedProcess[str]:
        """Run a fresh outside-checkout process, optionally injecting HTTP and interruption.

        Args:
            destination: Existing directory containing copied inputs and the process fixture.
            arguments: Non-secret command arguments interpreted relative to that directory.
            point: Synthetic fault boundary, or None to execute the installed command directly.

        Returns:
            Captured process exit, stdout progress and stderr diagnostics.
        """
        if point is None:
            command = [str(command_script), *arguments]
        else:
            command = [sys.executable, "-I", str(destination / "acquire.py"), str(destination),
                       str(destination / "run.json"), point, "--command", *arguments]
        return subprocess.run(command, cwd=destination, capture_output=True, text=True, timeout=30, check=False)


command_rows = []
command_failures = []
command_progress = []
with TemporaryDirectory(prefix="mr07-notebook-") as directory:
    command_root = Path(directory)
    for point, expected_status in (("after-raw", 75), ("interrupt", 130), ("exhausted", 4)):
        destination = command_root / point
        (destination / "raw").mkdir(parents=True)
        configuration = Configuration.read(root / "configs/data-jobs/fixture-inspection.json")
        job = replace(configuration.jobs[0], mode="historical", pipeline="acquisition.fetch")
        job = replace(job, end=job.start + timedelta(minutes=3))
        configuration = replace(configuration, jobs=(job,))
        (destination / "jobs.json").write_bytes(domain_codec.encode_json(configuration))
        shutil.copyfile(root / "configs/reference/binance-spot.json", destination / "catalog.json")
        shutil.copyfile(root / "tests/fixtures/mr06/acquire.py", destination / "acquire.py")
        command_run = configuration.resolve(job.name, catalog)
        (destination / "run.json").write_bytes(domain_codec.encode_json(command_run))
        arguments = ["--config", "jobs.json", "--catalog", "catalog.json", "--job", job.name,
                     "--manifest", "manifest.json", "--store", "raw",
                     "--rights-reference", job.rights_reference, "--timeout-seconds", "1",
                     "--max-bytes", "4096", "--retry-delays", "0", "0"]
        interrupted = NotebookAcquisitionCommand.process(destination, arguments, point)
        assert interrupted.returncode == expected_status, interrupted.stderr
        records = [json.loads(line) for line in interrupted.stdout.splitlines()]
        terminal_before = json.loads(interrupted.stderr) if interrupted.stderr else records[-1]
        assert terminal_before["completed"] == 1
        prefix = domain_codec.decode_json((destination / "manifest.json").read_bytes(), AcquisitionManifest)
        resumed = NotebookAcquisitionCommand.process(destination, arguments, "retry" if point == "interrupt" else "none")
        assert resumed.returncode == 0 and not resumed.stderr
        resume_records = [json.loads(line) for line in resumed.stdout.splitlines()]
        completed = resume_records[-1]
        assert completed["status"] == "complete" and completed["completed"] == completed["total"] == 3
        assert completed["command_identity"] == records[0]["command_identity"]
        saved_manifest = (destination / "manifest.json").read_bytes()
        state = domain_codec.decode_json(saved_manifest, AcquisitionManifest)
        assert state.completed[:len(prefix.completed)] == prefix.completed
        fetch_log = (destination / "fetches.txt").read_bytes()
        repeated = NotebookAcquisitionCommand.process(destination, arguments, None)
        assert repeated.returncode == 0 and not repeated.stderr
        assert (destination / "manifest.json").read_bytes() == saved_manifest
        assert (destination / "fetches.txt").read_bytes() == fetch_log
        command_rows.append((point, interrupted.returncode, terminal_before["completed"],
                             terminal_before["retries"], completed["completed"], completed["persisted_bytes"],
                             completed["last_persistence"], "unchanged; no fetch"))
        if point == "interrupt":
            command_progress = resume_records
    for damage in ("mutation", "missing"):
        damaged = destination / "raw" / state.completed[0].sha256
        original_payload = damaged.read_bytes()
        if damage == "mutation":
            damaged.chmod(0o644)
            damaged.write_bytes(b"X" + original_payload[1:])
        else:
            damaged.unlink()
        rejected = NotebookAcquisitionCommand.process(destination, arguments, None)
        assert rejected.returncode == (3 if damage == "mutation" else 4)
        error = json.loads(rejected.stderr)
        assert error["stage"] == "resume" and error["completed"] is None
        assert (destination / "manifest.json").read_bytes() == saved_manifest
        command_failures.append((damage, rejected.returncode, error["error_type"], error["completed"], "retained; rejected before fetch"))
        damaged.write_bytes(original_payload)
    command_identity = completed["command_identity"]
    command_run_identity = completed["run_identity"]

NotebookDisplay.show_table(
    ("Interruption", "Exit", "Completed before", "Retries before", "After resume", "Durable bytes", "Last acknowledged ingestion", "Repeat"),
    command_rows,
)
NotebookDisplay.show_table(("Rejected input", "Exit", "Error", "Verified completions", "Manifest"), command_failures)
NotebookDisplay.show_table(("Identity", "Value"), [("Resolved job/config", command_run_identity), ("Saved command", command_identity)])


Interruption,Exit,Completed before,Retries before,After resume,Durable bytes,Last acknowledged ingestion,Repeat
after-raw,75,1,0,3,132,2026-09-15T09:59:06.034721Z,unchanged; no fetch
interrupt,130,1,0,3,132,2026-09-15T09:59:07.559639Z,unchanged; no fetch
exhausted,4,1,2,3,132,2026-09-15T09:59:09.195993Z,unchanged; no fetch


Rejected input,Exit,Error,Verified completions,Manifest
mutation,3,ValueError,None,retained; rejected before fetch
missing,4,FileNotFoundError,None,retained; rejected before fetch


Identity,Value
Resolved job/config,7cd4cdd5476bdecf65e36f3fc101ab8010f6e04117df0c4891a0a5ed8b76a778
Saved command,7492b5c921ad63fef92a51f3d6bf912e614d33beeb8d7803f94dd534141fa4fe


In [19]:
NotebookDisplay.show_table(("Scope", "Stage", "Status", "Completed", "Retries", "Partition bytes", "Acknowledged bytes"), [
    (row["scope"], row["stage"], row["status"], row["completed"], row["retries"], row.get("bytes", "—"), row["persisted_bytes"])
    for row in command_progress
])
assert any(row["status"] == "retry" for row in command_progress)
assert command_progress[-1]["retries"] == 1
print("NB 07: installed entry point, flushed progress, interruption/resume, repeat and damaged-object rejection verified.")


Scope,Stage,Status,Completed,Retries,Partition bytes,Acknowledged bytes
job,configuration,started,None,0,—,None
job,resume,verifying,None,0,—,None
job,resume,resumed,1,0,—,44
partition,fetch,started,1,0,0,44
partition,fetch,failed,1,0,0,44
partition,fetch,retry,1,1,—,44
partition,fetch,started,1,1,0,44
partition,store,received,1,1,44,44
partition,store,stored,1,1,44,44
partition,checkpoint,checkpointing,1,1,—,44


NB 07: installed entry point, flushed progress, interruption/resume, repeat and damaged-object rejection verified.


### MR 07 findings and limits

The command prints flushed JSON progress before process exit. Job totals advance only after a durable completion checkpoint; partition `stored` alone does not add a completion. Resume reports its verified prefix, accepted compressed bytes and the original ingestion instant of the last acknowledged completion. Retry totals restart per invocation. A repeat of the same resolved input preserves the manifest and performs no source access.

Mutation and missing-byte failures report unknown verified totals while resume verification is incomplete. An abrupt process kill cannot emit a terminal record; its last flushed progress and saved checkpoint remain inspectable. Keyboard interruption returns 130; exhausted retries return 4 with the prior checkpoint. The saved command hash identifies resolved input, absolute destinations and non-secret limits, and stays stable across repeated invocations. Original completed clocks remain fixed; new/recovered ingestion uses the actual UTC clock.

This is synthetic installed-consumer evidence. Section 16 provides the real-download workflow. Actual source acceptance remains open until its configured run is performed and verified. No source coverage, semantic frame validation, accepted dataset or M1/P1 completion is claimed. See the [MR 07 record](../docs/roadmap/mr-07-implementation.md). Next is **MR 08 — Read saved raw frames with explicit gaps**; it is not implemented here.

Section 16 now exercises actual Binance archive acquisition; the Tardis-specific access boundary is superseded for P1.


## 16. Download actual Binance historical trades (NB 07 source follow-up)

The founder selected Binance public **individual trade archives** on 2026-09-15. This section uses real HTTPS through the installed `alpha-acquire` command, with no mocked transport and no API key. The [source contract](../docs/research/p1-baseline.md) keeps BTCUSDT and `[2024-06-01 00:00, 01:00)` UTC. Binance publishes a whole day in one ZIP, so raw storage includes the rest of June 1; the inspection below selects only the frozen hour.

**Run All downloads on the first run and verifies/reuses the saved object on repeats.** The user-selected location is `data/binance-archives/`, ignored by Git. Config, manifest and attempt logs live there beside a shared raw store. This is a local working copy with no automatic expiry; it is not a long-term archive or redistribution authorization. Changing code or inputs creates a separately identified run. Use `RUN_REAL_BINANCE=False` below (or `ALPHA_BINANCE_DOWNLOAD=0`) for offline foundations only.

The public [archive documentation](https://github.com/binance/binance-public-data) describes ZIP/CSV and SHA-256 sidecars. The transport checks both the publisher's checksum and complete ZIP integrity. That verifies downloaded bytes, not historical completeness or point-in-time availability. No Tardis capture clock or disconnect marker is invented.

In [20]:
RUN_REAL_BINANCE: bool = os.environ.get("ALPHA_BINANCE_DOWNLOAD", "1") == "1"
BINANCE_DIRECTORY = Path(os.environ.get("ALPHA_BINANCE_DIRECTORY", str(root / "data/binance-archives"))).expanduser().resolve()
BINANCE_MAX_BYTES = 64 * 1024 * 1024  # Separate caps for compressed ZIP and expanded CSV.

### Resolve and retain the actual inputs

The archive catalog is a new source binding of the existing authored assumptions. It does not turn 2026 metadata into 2024 eligibility evidence. The source request covers a UTC day; the resolved job preserves the selected hour. Inputs are saved before network access; a conflicting saved input fails instead of being overwritten.

In [21]:
# standard library imports
import sys
from contextlib import redirect_stderr, redirect_stdout
from pathlib import Path
from typing import TextIO
from uuid import uuid4

# project library imports
from alpha_research.configuration import Configuration
from alpha_research.data.manifests import AcquisitionManifest
from alpha_research.data.raw_store import RawStore
from alpha_research.data.requests import SourceRequest
from alpha_research.pipelines.acquisition import main as acquisition_main
from alpha_research.reference_data import CatalogSnapshot
from alpha_research.serialization import DomainCodec


class _NotebookArchiveFiles:
    """Provision local acquisition directories and preserve exact configuration on repeat runs."""

    @staticmethod
    def directory(destination: Path) -> None:
        """Create one private directory in an existing parent and synchronize its entry."""
        destination.mkdir(mode=0o700, exist_ok=True)
        descriptor = os.open(destination.parent, os.O_RDONLY | os.O_DIRECTORY)
        try:
            os.fsync(descriptor)
        finally:
            os.close(descriptor)

    @staticmethod
    def save(destination: Path, payload: bytes) -> None:
        """Preserve existing inputs and fail if their content differs from this selection."""
        try:
            with destination.open("xb") as output:
                output.write(payload)
                output.flush()
                os.fsync(output.fileno())
        except FileExistsError:
            if destination.read_bytes() != payload:
                raise ValueError(f"Saved input differs; choose a new acquisition directory: {destination.name}")


class _NotebookArchiveLog:
    """Keep command diagnostics outside Git while forwarding progress to the notebook."""

    def __init__(self, saved: TextIO, display: TextIO) -> None:
        """Bind the current attempt's file and the notebook's existing output stream."""
        self._saved = saved
        self._display = display

    def write(self, value: str) -> int:
        """Forward safe command output; downloaded market messages never enter this sink."""
        self._saved.write(value)
        self._display.write(value)
        self.flush()
        return len(value)

    def flush(self) -> None:
        """Make progress visible in both destinations as it arrives."""
        self._saved.flush()
        self._display.flush()


real_binance_arguments: list[str] | None = None
real_binance_directory: Path | None = None
real_codec = DomainCodec()
if RUN_REAL_BINANCE:
    if BINANCE_DIRECTORY.is_relative_to(root.resolve()) and not BINANCE_DIRECTORY.is_relative_to((root / "data/binance-archives").resolve()):
        raise ValueError("Choose the ignored data/binance-archives directory or storage outside the checkout.")
    real_configuration = Configuration.read(root / "configs/data-jobs/binance-archive.json")
    real_catalog = real_codec.decode_json((root / "configs/reference/binance-archive-spot.json").read_bytes(), CatalogSnapshot)
    real_run = real_configuration.resolve("btc-archive", real_catalog)
    for directory in (BINANCE_DIRECTORY, BINANCE_DIRECTORY / "raw", BINANCE_DIRECTORY / "runs"):
        _NotebookArchiveFiles.directory(directory)
    real_binance_directory = BINANCE_DIRECTORY / "runs" / real_run.identity
    _NotebookArchiveFiles.directory(real_binance_directory)
    for filename, value in (
        ("jobs.json", real_configuration), ("catalog.json", real_catalog),
        ("use-and-retention.json", {"rights_reference": real_run.job.rights_reference,
          "source_reference": "https://github.com/binance/binance-public-data",
          "reviewed_on": "2026-09-15", "intended_use": "private offline engineering/research",
          "retention": "User-selected local working copy; no automatic expiry; long-term archival rights not established."}),
    ):
        _NotebookArchiveFiles.save(real_binance_directory / filename, real_codec.encode_json(value))
    _NotebookArchiveFiles.directory(real_binance_directory / "attempts")
    real_binance_arguments = [
        "--config", str(real_binance_directory / "jobs.json"),
        "--catalog", str(real_binance_directory / "catalog.json"), "--job", real_run.job.name,
        "--manifest", str(real_binance_directory / "manifest.json"), "--store", str(BINANCE_DIRECTORY / "raw"),
        "--rights-reference", real_run.job.rights_reference, "--timeout-seconds", "10",
        "--max-bytes", str(BINANCE_MAX_BYTES), "--retry-delays", "1", "2",
    ]
    real_request = SourceRequest.partition(real_run)[0]
    NotebookDisplay.show_table(("Prepared actual request", "Value"), [
        ("Instrument / provider", f"{real_run.instruments[0].symbol} / {real_run.job.provider}"),
        ("Research interval", f"{real_run.job.start} to {real_run.job.end}"),
        ("Downloaded object interval", f"{real_request.start} to {real_request.end}"),
        ("Raw destination", BINANCE_DIRECTORY / "raw"), ("Run identity", real_run.identity),
        ("Time basis", real_run.job.capture_policy),
    ])
else:
    print("Actual Binance acquisition disabled for this run; set RUN_REAL_BINANCE=True to download or verify saved bytes.")

Prepared actual request,Value
Instrument / provider,BTCUSDT / binance-archive
Research interval,2024-06-01 00:00:00+00:00 to 2024-06-01 01:00:00+00:00
Downloaded object interval,2024-06-01 00:00:00+00:00 to 2024-06-02 00:00:00+00:00
Raw destination,/Users/lxg/Documents/GitHub/Alpha Research/alpha-research-lab-private/data/binance-archives/raw
Run identity,acb948277201cc169d32c2bb874fb40ce9e9c33c893e56a992faa47853147f80
Time basis,exchange-time-only-v1


### Download, interrupt, resume, or verify a completed run

This invokes the installed command's entry function with real HTTPS. Progress is flushed to the notebook and a separate attempt log. The 10-second socket timeout and two transient retries are explicit. The 64 MiB cap applies separately to ZIP and expanded bytes. Interrupt the cell to stop; rerun it with the same inputs to verify/resume. A completed run performs no download. Source mutation/missing stored bytes fail instead of silently replacing history.

In [22]:
real_binance_exit: int | None = None
if real_binance_arguments is not None and real_binance_directory is not None:
    attempt_path = real_binance_directory / "attempts" / f"{uuid4()}.jsonl"
    with attempt_path.open("x", encoding="utf-8") as attempt:
        progress = _NotebookArchiveLog(attempt, sys.stdout)
        with redirect_stdout(progress), redirect_stderr(progress):
            real_binance_exit = acquisition_main(real_binance_arguments)
        os.fsync(attempt.fileno())
    print(f"Acquisition exit: {real_binance_exit}; retained log: {attempt_path}")
    if real_binance_exit != 0:
        raise RuntimeError("Acquisition stopped. Inspect the reported failure and rerun this cell to verify/resume; retain the manifest and raw files.")
else:
    print("No real request was made. Prepare the enabled settings in the preceding cell first.")

{"command":{"api_key_env":null,"command":"alpha-acquire","manifest":"/Users/lxg/Documents/GitHub/Alpha Research/alpha-research-lab-private/data/binance-archives/runs/acb948277201cc169d32c2bb874fb40ce9e9c33c893e56a992faa47853147f80/manifest.json","max_bytes":67108864,"retry_delays":[1.0,2.0],"rights_reference":"binance-public-data-2026-09-15-private-research","run_identity":"acb948277201cc169d32c2bb874fb40ce9e9c33c893e56a992faa47853147f80","store":"/Users/lxg/Documents/GitHub/Alpha Research/alpha-research-lab-private/data/binance-archives/raw","timeout_seconds":10.0},"command_identity":"19a17a55325d95bffd25cb95f8bda2eae32dfed986e3995de8fee3c1cf8e8839","completed":null,"last_persistence":null,"persisted_bytes":null,"request_identity":null,"retries":0,"run_identity":"acb948277201cc169d32c2bb874fb40ce9e9c33c893e56a992faa47853147f80","scope":"job","stage":"configuration","status":"started","total":null}

{"command_identity":"19a17a55325d95bffd25cb95f8bda2eae32dfed986e3995de8fee3c1cf8e8839","completed":null,"last_persistence":null,"persisted_bytes":null,"request_identity":null,"retries":0,"run_identity":"acb948277201cc169d32c2bb874fb40ce9e9c33c893e56a992faa47853147f80","scope":"job","stage":"resume","status":"verifying","total":null}

{"command_identity":"19a17a55325d95bffd25cb95f8bda2eae32dfed986e3995de8fee3c1cf8e8839","completed":1,"last_persistence":"2026-09-15T09:46:04.125052Z","persisted_bytes":7308819,"request_identity":null,"retries":0,"run_identity":"acb948277201cc169d32c2bb874fb40ce9e9c33c893e56a992faa47853147f80","scope":"job","stage":"resume","status":"resumed","total":1}

{"command_identity":"19a17a55325d95bffd25cb95f8bda2eae32dfed986e3995de8fee3c1cf8e8839","completed":1,"exit_code":0,"last_persistence":"2026-09-15T09:46:04.125052Z","persisted_bytes":7308819,"request_identity":null,"retries":0,"run_identity":"acb948277201cc169d32c2bb874fb40ce9e9c33c893e56a992faa47853147f80","scope":"job","stage":"complete","status":"complete","total":1}

Acquisition exit: 0; retained log: /Users/lxg/Documents/GitHub/Alpha Research/alpha-research-lab-private/data/binance-archives/runs/acb948277201cc169d32c2bb874fb40ce9e9c33c893e56a992faa47853147f80/attempts/7041cb4c-1460-4939-ba3e-6fd8b14c29dd.jsonl


### Inspect real saved trades within the selected hour

Reopen the manifest and hash-check the stored ZIP. This notebook-only feasibility inspection streams the CSV, preserves decimal text, and reports count and boundary timestamps for the selected hour. It does not publish canonical events or implement MR 08's reusable reader. The 2024 archive uses Unix **milliseconds**; Binance documents microseconds for spot archives from 2025-01-01, which later readers must handle explicitly.

Local receipt/ingestion record today's acquisition. Recorder receipt and historical knowledge remain **unknown**. The raw bytes may be a later publisher revision; a checksum supplies no revision history. No coverage or causal backtest claim follows from this download.

In [23]:
# standard library imports
import csv
from io import TextIOWrapper
from zipfile import ZipFile

if real_binance_directory is not None:
    real_store = RawStore(BINANCE_DIRECTORY / "raw")
    real_checkpoint = AcquisitionManifest.read(real_binance_directory / "manifest.json", run=real_run, store=real_store)
    assert len(real_checkpoint.completed) == 1 and real_checkpoint.pending is None
    descriptor = real_checkpoint.completed[0]
    NotebookDisplay.show_table(("Exact ZIP SHA-256", "Bytes", "Original receipt", "Durable ingestion"), [
        (descriptor.sha256, descriptor.size_bytes, descriptor.ts_received, descriptor.ts_ingested)
    ])
    selected_count, archive_count = 0, 0
    selected_first, selected_last = None, None
    selected_preview = []
    epoch = datetime(1970, 1, 1, tzinfo=UTC)
    timestamp_unit = "microseconds" if real_request.start >= datetime(2025, 1, 1, tzinfo=UTC) else "milliseconds"
    with real_store.open(descriptor=descriptor, request=real_request) as saved:
        with ZipFile(saved) as archive, archive.open(archive.infolist()[0]) as member:
            for row in csv.reader(TextIOWrapper(member, encoding="utf-8", newline="")):
                if len(row) != 7 or not row[0].isascii() or not row[0].isdecimal() or not row[4].isascii() or not row[4].isdecimal():
                    raise ValueError("Unexpected Binance individual-trade CSV schema or integer timestamp")
                timestamp = epoch + timedelta(**{timestamp_unit: int(row[4])})
                if not real_request.start <= timestamp < real_request.end:
                    raise ValueError("Trade timestamp is outside the requested archive day")
                archive_count += 1
                if real_run.job.start <= timestamp < real_run.job.end:
                    selected_count += 1
                    selected_first = timestamp if selected_first is None else min(selected_first, timestamp)
                    selected_last = timestamp if selected_last is None else max(selected_last, timestamp)
                    if len(selected_preview) < 3:
                        selected_preview.append((row[0], timestamp, row[1], row[2]))
    assert selected_count > 0
    NotebookDisplay.show_table(("Observation", "Value"), [
        ("Archive rows", archive_count), ("Selected-hour rows", selected_count),
        ("First / last selected event", f"{selected_first} / {selected_last}"),
        ("Exchange timestamp unit", timestamp_unit), ("Recorder / historical knowledge", "not supplied / unknown"),
    ])
    NotebookDisplay.show_table(("Actual trade ID", "Exchange event time", "Price text (USDT/BTC)", "Quantity text (BTC)"), selected_preview)
    print("Real ZIP verified and selected hour inspected. Rerunning acquisition reuses this saved object. Canonical dataset/replay remains later roadmap work.")
else:
    print("No actual-source verification performed in this offline run.")

Exact ZIP SHA-256,Bytes,Original receipt,Durable ingestion
c9c333f58fe33faaa0a34d526c61b13f5648b908c4ad341dd6556e15c5bcc46a,7308819,2026-09-15 09:46:04.103212+00:00,2026-09-15 09:46:04.125052+00:00


Observation,Value
Archive rows,638484
Selected-hour rows,27480
First / last selected event,2024-06-01 00:00:00.001000+00:00 / 2024-06-01 00:59:59.997000+00:00
Exchange timestamp unit,milliseconds
Recorder / historical knowledge,not supplied / unknown


Actual trade ID,Exchange event time,Price text (USDT/BTC),Quantity text (BTC)
3620989757,2024-06-01 00:00:00.001000+00:00,67540.01000000,0.00099000
3620989758,2024-06-01 00:00:00.008000+00:00,67540.01000000,0.00071000
3620989759,2024-06-01 00:00:00.414000+00:00,67540.01000000,0.00118000


Real ZIP verified and selected hour inspected. Rerunning acquisition reuses this saved object. Canonical dataset/replay remains later roadmap work.


### Required failure examples on copies of actual bytes

The network download above used real HTTPS. These deliberately injected validation failures below make no network request and never modify the retained archive: a changed local object must fail its descriptor, and a wrong publisher digest must fail transport admission. Existing NB 07 also demonstrates missing objects, interruption, exhausted retries and recovery through the installed command.


In [24]:
# project library imports
from alpha_research.data.providers.binance.transport import BinanceArchiveTransport

if real_binance_directory is not None:
    real_failure_rows = []
    with real_store.open(descriptor=descriptor, request=real_request) as saved:
        verified_zip = saved.read()
    with TemporaryDirectory(prefix="binance-rejection-") as temporary:
        changed = Path(temporary) / "changed.zip"
        changed.write_bytes(b"X" + verified_zip[1:])
        try:
            descriptor.verify(changed, request=real_request)
        except ValueError as error:
            real_failure_rows.append(("Changed real ZIP copy", type(error).__name__, str(error)))
        else:
            raise AssertionError("Changed source bytes were accepted")
    bad_checksum = b"0" * 64 + f"  {real_request.instrument.symbol}-trades-{real_request.start.date()}.zip\n".encode()
    with patch.object(BinanceArchiveTransport, "_get", side_effect=[bad_checksum, verified_zip]):
        try:
            BinanceArchiveTransport(HTTPSConnection("data.binance.vision"), timeout_seconds=10, max_bytes=BINANCE_MAX_BYTES).fetch(
                real_request, permitted_rights_reference=real_run.job.rights_reference, api_key=None,
            )
        except ValueError as error:
            real_failure_rows.append(("Injected wrong publisher checksum", type(error).__name__, str(error)))
        else:
            raise AssertionError("A mismatched publisher checksum was accepted")
    NotebookDisplay.show_table(("Rejected case", "Failure", "Diagnostic"), real_failure_rows)
    del verified_zip
else:
    print("Actual-source failure examples skipped in this offline run.")


Rejected case,Failure,Diagnostic
Changed real ZIP copy,ValueError,"raw object SHA-256 mismatch: expected c9c333f58fe33faaa0a34d526c61b13f5648b908c4ad341dd6556e15c5bcc46a, got e958b2a046f0213ec8486b1f401468a4c395bf78669ffb354bfd76c1ea6f8300"
Injected wrong publisher checksum,ValueError,Binance archive SHA-256 differs from publisher checksum
